## Aim
The aim of this model is to allow for the creation of arbitrary molecules, e.g. the elements will be hardcoded but molecules will not be. The model doesn't need to be completely accurate; I want to incorporate a simplified simulation that still encompasses the interesting bits from chemistry such as distillation, reactions, etc. To add to the complexity, mods should be able to define how their materials are composed and be able to modify things without breaking the whole mod. Ideally this should be achieved via datapacks.

## Elements
The elements used will not be real life elements, but instead elements inspired by alchemy. This system will utilise the cardinal elements (Terra, Aqua, Aer, Ignis) and the tria prima (Salt, Sulphur, Mercury). Cardinal elements cannot bond to themselves, for example terra cannot bond to terra, aqua, aer, nor ignis. The tria prima are able to bond to themselves, and are also able to form a core that the cardinal elements can form a shell around (including different types). For example, the core can consist of salt and mercury, with a shell of aqua and ignis. The shells form in order of elements, with terra being closest to the core, and ignis the furthest away.

A core with a higher elemental shell will have greater difficulty bonding to a lower one, e.g. a salt core with aer shell will find it very difficult to bond to aqua, and even more difficult to bond to terra, but easily bond with ignis.

## Cosmology
The tria prima correspond to each of the 3 dimensions in Minecraft.
- Salt - Overworld
- Sulphur - Nether
- Mercury - End

If Mojang were to add a Sculk dimension, then this would correspond to Salt and the Overworld would instead be the interstice between those 3 dimensions. The matter in each of these dimensions would primarily have cores of their corresponding tria prima, e.g. Endstone would be made of something with a core of mercury plus some other cardinal elements.

The Sun and Moon correspond to Sulphur and Mercury respectively. "Singing" to the Sun and Moon will cause their rays to transform and influence chemical reactions. Some reactions will become easier depending on the time of day, phase of moon, and location.

## Metals
Gold, Iron, and Silver correspond to Sulphur, Salt, and Mercury in their perfected, crystalline forms. The properties of metals change as they come closer to their affinities, e.g. the strength of gold/silver depends on the position of the Sun/Moon and is stronger in the Nether/End, silver is affected by the phase of the moon, iron becomes stronger deeper underground, etc.

## Transmutation
Trees should be able to transmute the air and water into mass to build their trunks, and the player should be able to replicate this process chemically.

Netherrack should be able to burn indefinitely by transmuting the air into fire, though this flame will not be as hot as other means.

In [13]:
from matplotlib import pyplot as plt
from matplotlib.path import Path
import matplotlib.patches as patches

# Maximum number of elements in a block must fit in a 64-bit unsigned integer (2^64-1)
# but ideally should be a multiple of 16^3 so it can fit in the voxel of a 16x16x16 block
ELEMENTS_PER_BLOCK = 18_446_744_073_709_547_520  # (2^64 / 16^3 - 1) * 16^3
ELEMENTS_PER_VOXEL = 4_503_599_627_370_495  # 2^64 / 16^3 - 1

T_MAX = 750  # Kelvin
P_MAX = 1e12  # Pascal

T_ROOM = 295  # Kelvin
P_ATMO = 1e5  # Pascal

In [14]:
class Phase:
    def __init__(self, PV_area: Path) -> None:
        self._PV_area: Path = PV_area

    @property
    def PV_area(self) -> Path:
        return self._PV_area

class Solid(Phase):
    def __init__(self, PV_area: Path, density: float) -> None:
        super().__init__(PV_area)
        self._density: float = density

class Liquid(Phase):
    def __init__(self, PV_area: Path, density: float) -> None:
        super().__init__(PV_area)
        self._density: float = density

class Gas(Phase):
    def __init__(self, PV_area: Path) -> None:
        super().__init__(PV_area)

In [15]:
class Element:
    def __init__(self, name: str, mass: int) -> None:
        self._name: str = name
        self._mass: int = mass

        self._phases: list[Phase] = []
    
    def add_phase(self, phase: Phase) -> None:
        self._phases.append(phase)
    
    @property
    def phases(self) -> list[Phase]:
        return self._phases

aqua: Element = Element('aqua', 4)
elements: list[Element] = [aqua,]

In [16]:
verts = (
    (200, 0),
    (220, 100),
    (275, 600),  # Triple point
    (460, 11e6),
    (650, 22e6),  # Critical point
    (T_MAX, 22e6),
    (T_MAX, 0),
    (200, 0),  # Close path
)

codes = (
    Path.MOVETO,
    Path.CURVE3,
    Path.CURVE3,  # Triple point
    Path.CURVE3,
    Path.CURVE3,  # Critical point
    Path.LINETO,
    Path.LINETO,
    Path.CLOSEPOLY,  # Close path
)

gas_water: Gas = Gas(Path(verts, codes))
aqua.add_phase(gas_water)

In [ ]:
fig, ax = plt.subplots()

for e in elements:
    for p in e.phases:
        patch = patches.PathPatch(p.PV_area, lw=1)
        ax.add_patch(patch)

ax.axvline(x=T_ROOM, color='r', linestyle='--')
ax.axhline(y=P_ATMO, color='r', linestyle='--')

ax.set_yscale('symlog')
ax.set_xlim(0, T_MAX)
ax.set_ylim(0, P_MAX)
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Pressure (Pa)')

plt.tight_layout()
plt.show()

In [ ]:
class Container:
    def __init__(self, length: int, width: int, height: int, temperature: int = T_ROOM, pressure: int = P_ATMO) -> None:
        self._l = length
        self._w = width
        self._h = height
        
        self._T = temperature
        self._P = pressure

        self._solids = []
        self._liquids = []
        self._gasses = []

    def add_element(self, element: Element, amount: int, temperature: int) -> None:
        if element is Solid:
            pass
        elif element is Liquid:
            self._liquids.append(element)
        elif element is Gas:
            pass

    def plot(self) -> None:
        fig, ax = plt.subplots()
        ax.set_xlim(0, self._w)
        ax.set_ylim(0, self._h)
        ax.set_xlabel('Width')
        ax.set_ylabel('Height')

        plt.tight_layout()
        plt.show()

c = Container(16, 16, 16)
c.add_element(aqua, ELEMENTS_PER_VOXEL, T_ROOM)
c.plot()